In [1]:
from pyspark.sql import SparkSession

# Initialize your Spark instance
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .getOrCreate()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

### Extraction 

In [4]:
df=(
  spark.read.format('csv').option('header','true').option('inferSchema','true').load('COVID_19.csv')
)
df.show(10)

+--------+----+----+----+-----+-------+-----------+------------+--------------------+--------------+----------+---------+------+---------+
|     uid|fips|iso2|iso3|code3| admin2|   latitude|   longitude|      province_state|country_region|      date|confirmed|deaths|recovered|
+--------+----+----+----+-----+-------+-----------+------------+--------------------+--------------+----------+---------+------+---------+
|      16|  60|  AS| ASM|   16|   NULL|    -14.271|    -170.132|      American Samoa|            US|2020-01-22|        0|     0|     NULL|
|     316|  66|  GU| GUM|  316|   NULL|    13.4443|    144.7937|                Guam|            US|2020-01-22|        0|     0|     NULL|
|     580|  69|  MP| MNP|  580|   NULL|    15.0979|    145.6739|Northern Mariana ...|            US|2020-01-22|        0|     0|     NULL|
|     630|  72|  PR| PRI|  630|   NULL|    18.2208|    -66.5901|         Puerto Rico|            US|2020-01-22|        0|     0|     NULL|
|     850|  78|  VI| VIR|  

### Transformation 

Phase 1 : Data Profelling - Bronze Layer

In [5]:
df.count()

450597

In [6]:
df.printSchema()

root
 |-- uid: integer (nullable = true)
 |-- fips: integer (nullable = true)
 |-- iso2: string (nullable = true)
 |-- iso3: string (nullable = true)
 |-- code3: integer (nullable = true)
 |-- admin2: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- province_state: string (nullable = true)
 |-- country_region: string (nullable = true)
 |-- date: date (nullable = true)
 |-- confirmed: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- recovered: integer (nullable = true)



In [7]:

df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

+-----+-----+-----+-----+-----+------+--------+---------+--------------+--------------+----+---------+------+---------+
|  uid| fips| iso2| iso3|code3|admin2|latitude|longitude|province_state|country_region|date|confirmed|deaths|recovered|
+-----+-----+-----+-----+-----+------+--------+---------+--------------+--------------+----+---------+------+---------+
|30573|31863|30573|30573|30573| 31476|   13932|    13932|         22575|             0|   0|        0|     0|   420024|
+-----+-----+-----+-----+-----+------+--------+---------+--------------+--------------+----+---------+------+---------+



In [8]:
df.distinct().count()

450597

In [9]:
df.selectExpr(
    "min(date)",
    "max(date)"
).show()

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2020-01-22|2020-05-29|
+----------+----------+



Phase 2 : Data Quality Check - Silver Layer

In [10]:
print('duplicate records:', df.count() - df.distinct().count())

duplicate records: 0


In [11]:
df.filter(
  (col('confirmed')<0) | (col('deaths')<0)
).count()

0

In [13]:
df.filter(
  col('deaths')>col('confirmed')
).count()

578

In [14]:
silver_df=df.drop_duplicates()

In [16]:
silver_df=silver_df.withColumn('recovered',coalesce(col('recovered'),lit(0)))
silver_df.show(5)

+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+
|     uid| fips|iso2|iso3|code3|    admin2|   latitude|   longitude|province_state|country_region|      date|confirmed|deaths|recovered|
+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+
|84006109| 6109|  US| USA|  840|  Tuolumne|38.02644018|-119.9525093|    California|            US|2020-01-22|        0|     0|        0|
|84019161|19161|  US| USA|  840|       Sac|42.38624071|-95.10547892|          Iowa|            US|2020-01-22|        0|     0|        0|
|84022011|22011|  US| USA|  840|Beauregard|30.64836518|-93.34173616|     Louisiana|            US|2020-01-22|        0|     0|        0|
|84029065|29065|  US| USA|  840|      Dent|37.60663134|-91.50790473|      Missouri|            US|2020-01-22|        0|     0|        0|
|84029163|29163|  US| USA|  840|      Pik

In [17]:
silver_df=silver_df.withColumn('active_cases',col('confirmed')-col('deaths')-col('recovered'))

In [23]:
silver_df = silver_df.withColumn(
    "death_rate",
    when(
        col("confirmed") > 0,
        round((col("deaths") / col("confirmed")) * 100, 2)
    ).otherwise(0)
)

In [24]:
silver_df.show(5)

+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+------------+----------+
|     uid| fips|iso2|iso3|code3|    admin2|   latitude|   longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|
+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+------------+----------+
|84006109| 6109|  US| USA|  840|  Tuolumne|38.02644018|-119.9525093|    California|            US|2020-01-22|        0|     0|        0|           0|       0.0|
|84019161|19161|  US| USA|  840|       Sac|42.38624071|-95.10547892|          Iowa|            US|2020-01-22|        0|     0|        0|           0|       0.0|
|84022011|22011|  US| USA|  840|Beauregard|30.64836518|-93.34173616|     Louisiana|            US|2020-01-22|        0|     0|        0|           0|       0.0|
|84029065|29065|  US| USA|  840|  

In [25]:
silver_df=silver_df.withColumn(
  'recovery_rate',
  when(
    col('confirmed')>0,
    round((col('recovered')/col('confirmed'))*100,2)
  ).otherwise(0)
  )

In [26]:
silver_df.show()

+--------+-----+----+----+-----+----------+-----------+-------------------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+
|     uid| fips|iso2|iso3|code3|    admin2|   latitude|          longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|recovery_rate|
+--------+-----+----+----+-----+----------+-----------+-------------------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+
|84006109| 6109|  US| USA|  840|  Tuolumne|38.02644018|       -119.9525093|    California|            US|2020-01-22|        0|     0|        0|           0|       0.0|          0.0|
|84019161|19161|  US| USA|  840|       Sac|42.38624071|       -95.10547892|          Iowa|            US|2020-01-22|        0|     0|        0|           0|       0.0|          0.0|
|84022011|22011|  US| USA|  840|Beauregard|30.64836518|       -93.34173616|     Louisiana|

In [27]:
# silver_df.write.mode("overwrite") \
#     .parquet("/mnt/silver/covid")

Phase 3 : extracting KPI's(Key Performance Indicator) - Gold Layer

In [28]:
countary_cases=silver_df.groupBy('country_region').agg(max('confirmed').alias('total_cases')).orderBy(col('total_cases').desc())

In [29]:
countary_cases.show(10)

+--------------+-----------+
|country_region|total_cases|
+--------------+-----------+
|            US|    1746019|
|        Brazil|     465166|
|        Russia|     387623|
|United Kingdom|     271222|
|         Spain|     238564|
|         Italy|     232248|
|        France|     183816|
|       Germany|     182922|
|         India|     173491|
|        Turkey|     162120|
+--------------+-----------+
only showing top 10 rows


In [30]:
countary_deaths=silver_df.groupBy('country_region').agg(max('deaths').alias('total_deaths')).orderBy(col('total_deaths').desc())

In [31]:
countary_deaths.show()

+--------------+------------+
|country_region|total_deaths|
+--------------+------------+
|            US|      102809|
|United Kingdom|       38161|
|         Italy|       33229|
|         Spain|       28752|
|        France|       28663|
|        Brazil|       27878|
|       Belgium|        9430|
|        Mexico|        9415|
|       Germany|        8504|
|          Iran|        7677|
|   Netherlands|        5931|
|         India|        4980|
|         China|        4512|
|        Turkey|        4489|
|        Russia|        4374|
|        Sweden|        4350|
|          Peru|        4099|
|       Ecuador|        3334|
|   Switzerland|        1919|
|       Ireland|        1645|
+--------------+------------+
only showing top 20 rows


In [32]:
from pyspark.sql.window import Window

In [33]:
gold_df=silver_df.withColumn('prev_day_cases',lag('confirmed').over(
  Window.partitionBy('country_region').orderBy('date')
))

In [34]:
gold_df=gold_df.withColumn(
  'daily_new_cases',col('confirmed')-col('prev_day_cases')
)

In [ ]:
gold_df.orderBy(
    col("daily_new_cases").desc()
).show()



+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+--------------+---------------+
| uid|fips|iso2|iso3|code3|admin2|latitude|longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|recovery_rate|prev_day_cases|daily_new_cases|
+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+--------------+---------------+
|NULL|NULL|NULL|NULL| NULL|  NULL| 37.0902| -95.7129|          NULL|            US|2020-05-29|  1746019|102809|   406446|     1236764|      5.89|        23.28|            67|        1745952|
|NULL|NULL|NULL|NULL| NULL|  NULL| 37.0902| -95.7129|          NULL|            US|2020-05-28|  1721753|101616|   399991|     1220146|       5.9|        23.23|           360|        1721393|
|NULL|NULL|NULL|NULL| NULL|  NULL| 37.0902| -

In [44]:
w = Window.orderBy(col("confirmed").desc())

rank_df = gold_df.withColumn(
    "country_rank",
    dense_rank().over(w)
)


In [45]:
rank_df.show()

+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+--------------+---------------+------------+
| uid|fips|iso2|iso3|code3|admin2|latitude|longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|recovery_rate|prev_day_cases|daily_new_cases|country_rank|
+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+--------------+---------------+------------+
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-22|        0|     0|        0|           0|       0.0|          0.0|          NULL|           NULL|        8541|
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-23|        0|     0|        0|           0|       0.0|          0.0|             0|          